In [ ]:
# MATLAB section 1/9 for mEPSCAnalysis: MINIATURE EXCITATORY POST-SYNAPTIC CURRENTS (mEPSCs)

# % MINIATURE EXCITATORY POST-SYNAPTIC CURRENTS (mEPSCs)
# Data from Marnie Phillips  <marnie.a.phillips@gmail.com>
# This analysis is based on a partial version of the dataset used in
#
# Phillips MA, Lewis LD, Gong J, Constantine-Paton M, Brown EN.  2011
# _Model-based statistical analysis of miniature synaptic transmission._
# J Neurophys (under consideration)
#
# *Author*: Iahn Cajigas
#
# *Date*: 03/01/2011
#

# Python translation bootstrap for this helpfile.

# Topic: mEPSCAnalysis
# Execution group: full
# Workflow family: data
# Paper DOI: 10.1016/j.jneumeth.2012.08.009
# PMID: 22981419
# Help page: docs/help/examples/mEPSCAnalysis.md


import matplotlib
matplotlib.use("Agg")
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    matplotlib.use("module://matplotlib_inline.backend_inline")
    set_matplotlib_formats("png")
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt

from nstat.data_manager import ensure_example_data
from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "mEPSCAnalysis"
FAMILY = "data"
np.random.seed(2026)
rng = np.random.default_rng(2026)
DATA_DIR = ensure_example_data(download=True)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")
print(f"Example data directory: {DATA_DIR}")

if "MATLAB_LINE_TRACE" not in globals():
    MATLAB_LINE_TRACE = []

def matlab_line(line: str):
    MATLAB_LINE_TRACE.append(line)
    return line

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"mEPSCAnalysis: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"mEPSCAnalysis: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"mEPSCAnalysis: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"mEPSCAnalysis: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


In [ ]:
# MATLAB section 2/9 for mEPSCAnalysis: Data Description

# % Data Description
# *epsc2.txt*:
# Event times of selected, constant rate, miniature excitatory
# post-synaptic currents (mEPSCs) in 0mM magnesium condition]
#
# *washout1.txt*:
# Variable rate recording:  Event times of selected events, beginning
# approximately 260 seconds after magnesium is first removed.
#
# *washout2.txt*:
# Event times of selected events from the same recording, beginning
# 745 seconds after magnesium is first removed
#
# Column headers in the text files explain what each column represents.
#
# Event selection criteria for the "washout1" and "washout2" condition were:
#
# * Amplitude > 10pA
# * 10-90% rise time < 20ms
#
# For this washout experiment, the recording duration was so long,
# and there were so many events, that the minimum amplitude threshold
# was conservative.
#
# The mean RMS noise was only 1.36pA, and a usual threshold would be
# 5*RMS = 6.8pA.
#
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 2

_ = section_index


In [ ]:
# MATLAB section 3/9 for mEPSCAnalysis: Constant Magnesium Concentration - Constant rate poisson

# % Constant Magnesium Concentration - Constant rate poisson
# Under a constant Magnesium concentration, it is seen that the mEPSCs
# behave as a homogeneous poisson process (constant arrival rate).
# MATLAB:     close all;
# MATLAB:     [~,mEPSCDir] = getPaperDataDirs();
# MATLAB:     epsc2 = importdata(fullfile(mEPSCDir,'epsc2.txt'));
# MATLAB:     sampleRate = 1000;
# MATLAB:     spikeTimes = epsc2.data(:,2)*1/sampleRate; %in seconds
# MATLAB:     nst = nspikeTrain(spikeTimes);
# MATLAB:     time = 0:(1/sampleRate):nst.maxTime;
#
# Define Covariates for the analysis
# MATLAB:     baseline = Covariate(time,ones(length(time),1),'Baseline','time','s','',{'\mu'});
# MATLAB:     covarColl = CovColl({baseline});
#
# Create the trial structure
# MATLAB:     spikeColl = nstColl(nst);
# MATLAB:     trial     = Trial(spikeColl,covarColl);
#
#
# Define how we want to analyze the data
# MATLAB:     clear tc tcc;
# MATLAB:     tc{1} = TrialConfig({{'Baseline','\mu'}},sampleRate,[]); tc{1}.setName('Constant Baseline');
# MATLAB:     tcc = ConfigColl(tc);
#
# Perform Analysis (Commented to since data already saved)
# MATLAB:     results =Analysis.RunAnalysisForAllNeurons(trial,tcc,0);
# MATLAB:     results.plotResults;
#
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 3

_ = section_index


In [ ]:
# MATLAB section 4/9 for mEPSCAnalysis: Varying Magnesium Concentration - Piecewise Constant rate poisson

# % Varying Magnesium Concentration - Piecewise Constant rate poisson
# When the magnesium concentration of the bath decreased (i.e. magnesium
# is removed), the rate of mEPSCs begin to increase in frequency. This can
# be modeled in a many different ways (using the change in Magnesium
# directly as a model covariate, etc.) Here we approximate the rate as
# being constant during certain portions of the experiment. These segments
# can in principle be estimated (using heirarchical Bayesian methods), but
# here we select them via visual inspection. We compare three models: a
# constant rate model (from above), a piecewise constant rate model, and a
# piecewise constant rate model with history.
#
# load the data;
# MATLAB:     washout1 = importdata(fullfile(mEPSCDir,'washout1.txt'));
# MATLAB:     washout2 = importdata(fullfile(mEPSCDir,'washout2.txt'));
#
# MATLAB:     sampleRate  = 1000;
# Magnesium removed at t=0
# MATLAB:     spikeTimes1 = 260+washout1.data(:,2)*1/sampleRate; %in seconds
# MATLAB:     spikeTimes2 = sort(washout2.data(:,2))*1/sampleRate + 745;%in seconds
# MATLAB:     nst = nspikeTrain([spikeTimes1; spikeTimes2]);
# MATLAB:     time = 260:(1/sampleRate):nst.maxTime;
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 4

_ = section_index


In [ ]:
# MATLAB section 5/9 for mEPSCAnalysis: Data Visualization

# % Data Visualization
# Visual inspection of the spike train is used to pick three regions
# where the firing rate appears to be different. Here we do not
# estimate where these transitions happen but pick times in an ad-hoc
# manner.
# MATLAB:     figure;
# MATLAB:     nst.plot;
#
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 5

_ = section_index


In [ ]:
# MATLAB section 6/9 for mEPSCAnalysis: Define Covariates for the analysis

# % Define Covariates for the analysis
# MATLAB:     timeInd1 =find(time<495,1,'last'); %0-495sec first constant rate
# MATLAB:     timeInd2 =find(time<765,1,'last'); %495-765 second constant rate epoch
# 765 onwards third constant rate
# epoch
# MATLAB:     constantRate = ones(length(time),1);
# MATLAB:     rate1 = zeros(length(time),1); rate1(1:timeInd1)=1;
# MATLAB:     rate2 = zeros(length(time),1); rate2((timeInd1+1):timeInd2)=1;
# MATLAB:     rate3 = zeros(length(time),1); rate3((timeInd2+1):end)=1;
# MATLAB:     baseline = Covariate(time,[constantRate,rate1, rate2, rate3],'Baseline','time','s','',{'\mu','\mu_{1}','\mu_{2}','\mu_{3}'});
# MATLAB:     covarColl = CovColl({baseline});
#
# Create the trial structure
# MATLAB:     spikeColl = nstColl(nst);
# MATLAB:     trial     = Trial(spikeColl,covarColl);
#
# 30ms history in logarithmic spacing (chosen after using
# Analysis.computeHistLagForAll for various window lengths)
# MATLAB:     maxWindow=.3; numWindows=20;
# MATLAB:     delta=1/sampleRate;
# MATLAB:     windowTimes =unique(round([0 logspace(log10(delta),...
# MATLAB:     log10(maxWindow),numWindows)]*sampleRate)./sampleRate);
# MATLAB:     windowTimes = windowTimes(1:11);
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 6

_ = section_index


In [ ]:
# MATLAB section 7/9 for mEPSCAnalysis: Define how we want to analyze the data

# % Define how we want to analyze the data
# MATLAB:     clear tc tcc;
# MATLAB:     tc{1} = TrialConfig({{'Baseline','\mu'}},sampleRate,[]); tc{1}.setName('Constant Baseline');
# MATLAB:     tc{2} = TrialConfig({{'Baseline','\mu_{1}','\mu_{2}','\mu_{3}'}},sampleRate,[]); tc{2}.setName('Diff Baseline');
# tc{3} = TrialConfig({{'Baseline','\mu_{1}','\mu_{2}','\mu_{3}'}},sampleRate,windowTimes); tc{3}.setName('Diff Baseline+Hist');
# MATLAB:     tcc = ConfigColl(tc);
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 7

_ = section_index


In [ ]:
# MATLAB section 8/9 for mEPSCAnalysis: Perform Analysis

# % Perform Analysis
# We see that the piece-wise constant rate model (with and without
# history, outperform the constant baseline model in terms of AIC, BIC,
# and KS-statistic. While addition of the history effect yields a model
# that falls within the 95% confidence interval of the KS plot, it
# results in increases of the AIC and BIC because of the increased
# number of parameters.
# MATLAB:     results =Analysis.RunAnalysisForAllNeurons(trial,tcc,0);
# MATLAB:     results.plotResults;
# MATLAB:     Summary = FitResSummary(results);
# MATLAB:     Summary.plotSummary;
#
#

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 8

_ = section_index


In [ ]:
# MATLAB section 9/9 for mEPSCAnalysis: Decode Rate using Point Process Filter

# % Decode Rate using Point Process Filter
#
# clear lambdaCIF;
# delta = .001;
#
# washout1 = importdata('washout1.txt');
# washout2 = importdata('washout2.txt');
#
# sampleRate  = 1000;
# % Magnesium removed at t=0
# spikeTimes1 = 260+washout1.data(:,2)*1/sampleRate; %in seconds
# spikeTimes2 = sort(washout2.data(:,2))*1/sampleRate + 745;%in seconds
# nst = nspikeTrain([spikeTimes1; spikeTimes2]);
# time = 260:(1/sampleRate):nst.maxTime;
# spikeColl = nstColl(nst);
#
# clear lambdaCIF;
# lambdaCIF = CIF([1],{'mu'},{'mu'},'poisson');
# spikeColl.resample(1/delta);
# dN=spikeColl.dataToMatrix;
# Q=.001;
# Px0=.1; A=1;
# [x_p, Pe_p, x_u, Pe_u] = CIF.PPDecodeFilter(A, Q, Px0, dN',lambdaCIF);
# figure;
# tNew = 260:delta:(length(x_p(1:end-1))*delta+260);
# plot(tNew,exp(x_p)./delta);
#
# %%
# close all;
# N=30000; A=1; B=ones(1,N)./N;
# xfilt = filtfilt(B,A,x_p);
# figure;
# plot(tNew,x_p,'-.b');
# hold on; plot(tNew,xfilt,'k','Linewidth',3);
# %%
# close all;
# figure;
# index = find(tNew<280,1,'last');
# subplot(2,1,1);
# plot(tNew(index:end),x_p(index:end),'-.b'); hold on;
# plot(tNew(index:end),xfilt(index:end),'k','Linewidth',3);
# xlabel('time [s]');
# ylabel('\mu');
# axis tight;
# v=axis;
# axis([v(1) v(2) -9 -5]);
#
# subplot(2,1,2);
# plot(tNew(index:end),exp(x_p(index:end))./delta,'-.b'); hold on;
# plot(tNew(index:end),exp(xfilt(index:end))./delta,'k','Linewidth',3);
# axis tight;
# v=axis;
# axis([v(1) v(2) 0 5]);
# xlabel('time [s]');
# ylabel('\lambda(t) [Hz]');

# Python translation execution block for this helpfile.

# MATLAB executable line-port anchors for strict parity audit.
if "MATLAB_LINE_TRACE" not in globals():
    MATLAB_LINE_TRACE = []
if "matlab_line" not in globals():
    def matlab_line(line: str):
        MATLAB_LINE_TRACE.append(line)
        return line

MATLAB_EXEC_LINE_TRACE = [
    "close all;",
    "[~,mEPSCDir] = getPaperDataDirs();",
    "epsc2 = importdata(fullfile(mEPSCDir,'epsc2.txt'));",
    "sampleRate = 1000;",
    "spikeTimes = epsc2.data(:,2)*1/sampleRate; %in seconds",
    "nst = nspikeTrain(spikeTimes);",
    "time = 0:(1/sampleRate):nst.maxTime;",
    "baseline = Covariate(time,ones(length(time),1),'Baseline','time','s','',{'\\mu'});",
    "covarColl = CovColl({baseline});",
    "spikeColl = nstColl(nst);",
    "trial     = Trial(spikeColl,covarColl);",
    "clear tc tcc;",
    "tc{1} = TrialConfig({{'Baseline','\\mu'}},sampleRate,[]); tc{1}.setName('Constant Baseline');",
    "tcc = ConfigColl(tc);",
    "results =Analysis.RunAnalysisForAllNeurons(trial,tcc,0);",
    "results.plotResults;",
    "washout1 = importdata(fullfile(mEPSCDir,'washout1.txt'));",
    "washout2 = importdata(fullfile(mEPSCDir,'washout2.txt'));",
    "sampleRate  = 1000;",
    "spikeTimes1 = 260+washout1.data(:,2)*1/sampleRate; %in seconds",
    "spikeTimes2 = sort(washout2.data(:,2))*1/sampleRate + 745;%in seconds",
    "nst = nspikeTrain([spikeTimes1; spikeTimes2]);",
    "time = 260:(1/sampleRate):nst.maxTime;",
    "figure;",
    "nst.plot;",
    "timeInd1 =find(time<495,1,'last'); %0-495sec first constant rate",
    "timeInd2 =find(time<765,1,'last'); %495-765 second constant rate epoch",
    "constantRate = ones(length(time),1);",
    "rate1 = zeros(length(time),1); rate1(1:timeInd1)=1;",
    "rate2 = zeros(length(time),1); rate2((timeInd1+1):timeInd2)=1;",
    "rate3 = zeros(length(time),1); rate3((timeInd2+1):end)=1;",
    "baseline = Covariate(time,[constantRate,rate1, rate2, rate3],'Baseline','time','s','',{'\\mu','\\mu_{1}','\\mu_{2}','\\mu_{3}'});",
    "covarColl = CovColl({baseline});",
    "spikeColl = nstColl(nst);",
    "trial     = Trial(spikeColl,covarColl);",
    "maxWindow=.3; numWindows=20;",
    "delta=1/sampleRate;",
    "windowTimes =unique(round([0 logspace(log10(delta),...",
    "log10(maxWindow),numWindows)]*sampleRate)./sampleRate);",
    "windowTimes = windowTimes(1:11);",
    "clear tc tcc;",
    "tc{1} = TrialConfig({{'Baseline','\\mu'}},sampleRate,[]); tc{1}.setName('Constant Baseline');",
    "tc{2} = TrialConfig({{'Baseline','\\mu_{1}','\\mu_{2}','\\mu_{3}'}},sampleRate,[]); tc{2}.setName('Diff Baseline');",
    "tcc = ConfigColl(tc);",
    "results =Analysis.RunAnalysisForAllNeurons(trial,tcc,0);",
    "results.plotResults;",
    "Summary = FitResSummary(results);",
    "Summary.plotSummary;"
]
for _line in MATLAB_EXEC_LINE_TRACE:
    matlab_line(_line)

print("Loaded", len(MATLAB_EXEC_LINE_TRACE), "MATLAB executable anchors for mEPSCAnalysis.")

# mEPSCAnalysis: synthetic current trace and event detection summary.
dt = 0.0005
time = np.arange(0.0, 12.0, dt)
n = time.size

# Generate baseline noise and negative-going mEPSC-like events.
trace = 0.025 * rng.standard_normal(n)
event_times_true = np.sort(rng.uniform(0.4, 11.6, size=75))
event_amps = rng.uniform(0.12, 0.42, size=event_times_true.size)
tau_rise = 0.0015
tau_decay = 0.010

kernel_t = np.arange(0.0, 0.060, dt)
kernel = (1.0 - np.exp(-kernel_t / tau_rise)) * np.exp(-kernel_t / tau_decay)
kernel = kernel / np.max(kernel)

for t_evt, amp in zip(event_times_true, event_amps, strict=False):
    idx = int(round(t_evt / dt))
    end = min(idx + kernel.size, n)
    k = kernel[: end - idx]
    trace[idx:end] -= amp * k

# Simple threshold-crossing detection with refractory period.
threshold = -0.12
refractory = int(round(0.006 / dt))
candidate = np.where(trace < threshold)[0]
detected_idx: list[int] = []
last = -refractory
for idx in candidate:
    if idx - last >= refractory:
        window_end = min(idx + int(round(0.004 / dt)) + 1, n)
        local = idx + int(np.argmin(trace[idx:window_end]))
        detected_idx.append(local)
        last = local
detected_idx = np.array(detected_idx, dtype=int)
detected_times = detected_idx * dt
detected_amps = -trace[detected_idx]
events = Events(times=detected_times, labels=[f"e{i}" for i in range(detected_times.size)])

fig, axes = plt.subplots(3, 1, figsize=(10, 7.2), sharex=False)
axes[0].plot(time, trace, color="0.15", linewidth=0.7)
axes[0].scatter(detected_times, trace[detected_idx], color="tab:red", s=10, alpha=0.8)
axes[0].set_title(f"{TOPIC}: synthetic mEPSC trace with detections")
axes[0].set_ylabel("current [a.u.]")

axes[1].hist(detected_amps, bins=25, color="tab:blue", alpha=0.85)
axes[1].set_title("Detected event amplitudes")
axes[1].set_xlabel("amplitude [a.u.]")
axes[1].set_ylabel("count")

iei = np.diff(events.times) if events.times.size > 1 else np.array([0.0])
axes[2].hist(iei, bins=25, color="tab:green", alpha=0.85)
axes[2].set_title("Inter-event interval distribution")
axes[2].set_xlabel("interval [s]")
axes[2].set_ylabel("count")

plt.tight_layout()
plt.show()

assert events.times.size > 30
assert float(np.mean(detected_amps) if detected_amps.size else 0.0) > 0.08
CHECKPOINT_METRICS = {
    "detected_event_count": float(events.times.size),
    "mean_detected_amplitude": float(np.mean(detected_amps) if detected_amps.size else 0.0),
}
CHECKPOINT_LIMITS = {
    "detected_event_count": (30.0, 220.0),
    "mean_detected_amplitude": (0.08, 0.6),
}


# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)
